# Heretic 云端破解 Qwen3.5-2B → 转 MNN（装回手机 App）
运行路径：**代码 → 更改运行时类型 → T4 GPU**，然后依次运行全部代码块。
第 1-4 块：云端破解（去审查）；第 5-8 块：破解模型转回 MNN 格式，下载 `llm.mnn` 替换手机 App。

In [ ]:
# 1. 安装 Heretic
!pip install -q -U heretic-llm
print('install ok')

In [ ]:
# 2. 写非交互配置（关键：export_strategy + model_action + save_directory 必须显式指定，否则卡确认）
config = '''
quantization = "none"          # T4 15GB 够跑 fp16
device_map = "auto"
offload_outputs_to_cpu = true
export_strategy = "merge"      # 导出完整模型
model_action = "save"          # 保存到本地（不传 HF）
save_directory = "output_model"  # 破解后的模型存这里
batch_size = 0
study_checkpoint_dir = "checkpoints"
modified_model_type = "decensored"
system_prompt = "You are a helpful assistant."
'''
with open('config.toml','w') as f: f.write(config)
print('config.toml written')

In [ ]:
# 3. 跑 Heretic 破解（去审查，默认动作）
!heretic Qwen/Qwen3.5-2B -c config.toml

In [ ]:
# 4. 确认破解模型输出
!ls -lh output_model/ | head -20
!find output_model -name '*.safetensors' | head

In [ ]:
# 5. 安装 MNN 转换依赖
!pip install -q MNN onnx onnxslim onnxruntime peft sentencepiece datasets yaspin
print('mnn deps ok')

In [ ]:
# 6. 拉 MNN 转换工具（只取 transformers/llm/export）
!git clone --depth 1 --filter=blob:none --sparse https://github.com/alibaba/MNN.git mnn
!cd mnn && git sparse-checkout set transformers/llm/export
print('mnn tool ok')

In [ ]:
# 7. 破解模型 → MNN 格式（默认 4bit，产出 llm.mnn）
!python mnn/transformers/llm/export/llmexport.py --path ./output_model --export mnn --dst_path ./mnn_out --quant_bit 4
!ls -lh mnn_out/

In [ ]:
# 8. 下载 llm.mnn（替换手机 App 的模型文件）
!cp mnn_out/llm.mnn ./llm_heretic.mnn
from google.colab import files
files.download('llm_heretic.mnn')
print('done')